# AI Home Mixologist

## Google Colab Teaching Notebook

This notebook builds an AI-powered home cocktail recommender step by step.

The project will demonstrate:

- loading and cleaning a cocktail dataset
- matching available home ingredients to recipes
- generating shopping lists
- estimating the best value ingredient to buy next
- converting recipes into text documents
- creating embeddings
- building a FAISS vector index
- manually implementing retrieval-augmented generation, or RAG
- prompting an LLM for personalized recommendations
- evaluating the recommendation system

**Notebook rule:** all project code stays inside this one notebook. No separate Python files are required.


# Section 1: Install Packages

Before we can build the mixologist system, we need to install and verify the Python libraries used throughout the notebook.

In later sections, we will use these packages for data handling, ingredient matching, embeddings, vector search, and LLM recommendations.

This section is intentionally simple because package setup should be easy to rerun in Google Colab.


## Why This Step Is Necessary

- `pandas` and `numpy` help us load, clean, and analyze recipe data.
- `datasets` lets us download the Hugging Face cocktail dataset directly in Colab.
- `sentence-transformers` turns recipe documents into embeddings.
- `faiss-cpu` lets us search embeddings efficiently.
- `transformers` and `accelerate` let us run a small teaching LLM in Colab.
- `ipywidgets` helps us create a simple interactive demo.

We will implement the RAG pipeline manually instead of using LangChain so each concept is visible and understandable.

In [1]:
# ============================================================
# Section 1: Install Packages
# ============================================================

# Install the libraries needed for the full notebook.
# The -q flag keeps the Colab output cleaner for beginners.
%pip install -q pandas numpy datasets sentence-transformers faiss-cpu transformers accelerate ipywidgets

# Import package metadata so we can confirm versions after installation.
import importlib.metadata as package_metadata

# Map display names to installed package names.
required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "datasets": "datasets",
    "sentence-transformers": "sentence-transformers",
    "faiss-cpu": "faiss-cpu",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "ipywidgets": "ipywidgets",
}

print("Installed package versions:")

for display_name, package_name in required_packages.items():
    package_version = package_metadata.version(package_name)
    print(f"- {display_name}: {package_version}")

print("\nAll required packages are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 50.0 MB/s eta 0:00:00
Installed package versions:
- pandas: 2.2.2
- numpy: 2.0.2
- datasets: 4.0.0
- sentence-transformers: 5.6.0
- faiss-cpu: 1.14.3
- transformers: 5.13.1
- accelerate: 1.14.0
- ipywidgets: 7.7.1

All required packages are ready.


## Example Output

Your exact version numbers may be different because Colab updates packages over time.

```text
Installed package versions:
- pandas: 2.x.x
- numpy: 1.x.x or 2.x.x
- datasets: 2.x.x or newer
- sentence-transformers: 3.x.x or newer
- faiss-cpu: 1.x.x
- transformers: 4.x.x or newer
- accelerate: 0.x.x or newer
- ipywidgets: 7.x.x or 8.x.x

All required packages are ready.
```

## Short Comments

- We install packages first so every later section can run without missing-library errors.
- Version checks make the notebook easier to debug if something behaves differently in the future.
- Keeping installation in one cell makes the notebook clean and beginner-friendly.
- No LangChain is installed because this notebook builds the RAG pipeline manually.

# Section 2: Load Dataset

In this section, we load the cocktail recipe dataset from Hugging Face.

We will use the dataset ID provided for this project:

```text
erwanlc/cocktails_recipe
```

The goal is to create one raw DataFrame called `cocktails_raw_df`. Later sections will explore, clean, and transform this DataFrame into a recommendation-ready format.


## Why This Step Is Necessary

A recommendation system starts with data.

Before we can match ingredients, build documents, generate embeddings, or retrieve recipes, we need to load the recipe dataset into memory.

This section also prints the dataset features so beginners can see what columns are available before using them.


In [2]:
# ============================================================
# Section 2: Load Dataset
# ============================================================

# Import libraries used for dataset loading and table handling.
from datasets import load_dataset
import pandas as pd


# Store the Hugging Face dataset ID and split in clear variables.
DATASET_ID = "erwanlc/cocktails_recipe"
DATASET_SPLIT = "train"


def load_huggingface_dataset(dataset_id, dataset_split):
    """Load a Hugging Face dataset split into memory."""
    return load_dataset(dataset_id, split=dataset_split)


def convert_dataset_to_dataframe(huggingface_dataset):
    """Convert a Hugging Face Dataset object into a pandas DataFrame."""
    return huggingface_dataset.to_pandas()


def print_loading_report(dataset_id, dataset_split, huggingface_dataset, dataframe):
    """Print a beginner-friendly summary of the loaded dataset."""
    print(f"Dataset ID: {dataset_id}")
    print(f"Dataset split: {dataset_split}")
    print(f"Rows: {dataframe.shape[0]}")
    print(f"Columns: {dataframe.shape[1]}")

    print("\nColumn names:")
    print(list(dataframe.columns))

    print("\nDataset features:")
    print(huggingface_dataset.features)


# Load the Hugging Face dataset split into the Colab environment.
cocktails_dataset = load_huggingface_dataset(DATASET_ID, DATASET_SPLIT)

# Convert the dataset to a pandas DataFrame for easier data analysis.
cocktails_raw_df = convert_dataset_to_dataframe(cocktails_dataset)

# Print a short loading report so we know the dataset is ready.
print_loading_report(DATASET_ID, DATASET_SPLIT, cocktails_dataset, cocktails_raw_df)

# Display the first few rows so we can visually confirm the data loaded correctly.
cocktails_raw_df.head()


README.md:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/2.50M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6956 [00:00<?, ? examples/s]

Dataset ID: erwanlc/cocktails_recipe
Dataset split: train
Rows: 6956
Columns: 5

Column names:
['title', 'glass', 'garnish', 'recipe', 'ingredients']

Dataset features:
{'title': Value('string'), 'glass': Value('string'), 'garnish': Value('string'), 'recipe': Value('string'), 'ingredients': Value('string')}


,title,glass,garnish,recipe,ingredients
0,Abacaxi Ricaço,Pineapple shell (frozen) glass,Cut a straw sized hole in the top of the pinea...,Cut the top off a small pineapple and carefull...,"[['1 whole', 'Pineapple (fresh)'], ['9 cl', 'H..."
1,Abbey,Coupe glass,Orange zest twist,SHAKE all ingredients with ice and fine strain...,"[['4.5 cl', 'Rutte Dry Gin'], ['2.25 cl', 'Lil..."
2,A.B.C. Cocktail,Nick & Nora glass,Lemon zest twist & Luxardo Maraschino cherry,TEAR mint and place in shaker. Add other ingre...,"[['7 fresh', 'Mint leaves'], ['3 cl', 'Tawny p..."
3,Absinthe Cocktail,Coupe glass,Mint leaf,SHAKE all ingredients with ice and fine strain...,"[['3 cl', 'La Fée Parisienne absinthe'], ['7.5..."
4,Absinthe Frappé,Old-fashioned glass,Mint sprig,SHAKE all ingredients with ice and fine strain...,"[['4.5 cl', 'La Fée Parisienne absinthe'], ['1..."


## Example Output

Your exact row count may differ if the Hugging Face dataset is updated.

```text
Dataset ID: erwanlc/cocktails_recipe
Dataset split: train
Rows: 6960
Columns: 5

Column names:
['title', 'glass', 'garnish', 'recipe', 'ingredients']

Dataset features:
{'title': Value('string'), 'glass': Value('string'), 'garnish': Value('string'), 'recipe': Value('string'), 'ingredients': Value('string')}
```

Below the text output, Colab will also display the first five rows of `cocktails_raw_df` as a table.


## Short Comments

- We use Hugging Face `datasets` so the notebook can download the dataset directly in Colab.
- We print the features before cleaning so the data structure is transparent.
- We load the raw data into `cocktails_raw_df` and keep it unchanged for now.
- Cleaning happens later so we can compare the original data with the cleaned version.


# Section 3: Explore Dataset

Now that the dataset is loaded, we need to understand what it contains before cleaning or modeling.

Exploratory data analysis, often called EDA, helps us answer basic questions:

- How many recipes are in the dataset?
- What columns are available?
- Which columns have missing values?
- Are there duplicate cocktail names?
- What do the ingredient strings look like?
- Which glass types appear most often?

This section does not modify the dataset. It only studies `cocktails_raw_df`.


## Why This Step Is Necessary

A mixology recommendation system depends heavily on clean recipe and ingredient data.

If we skip exploration, we might make wrong assumptions about column names, missing values, duplicate recipes, or ingredient formatting.

By inspecting the raw dataset first, we can make better decisions in the cleaning section.


In [3]:
# ============================================================
# Section 3: Explore Dataset
# ============================================================

# Import display so tables look clean in Google Colab.
from IPython.display import display

import pandas as pd


def show_dataset_shape(dataframe):
    """Print the number of rows and columns in a DataFrame."""
    row_count, column_count = dataframe.shape
    print(f"Rows: {row_count}")
    print(f"Columns: {column_count}")


def create_column_summary(dataframe):
    """Create a summary table for column types, missing values, and unique values."""
    summary_df = pd.DataFrame({
        "column_name": dataframe.columns,
        "data_type": [str(dataframe[column].dtype) for column in dataframe.columns],
        "missing_values": [dataframe[column].isna().sum() for column in dataframe.columns],
        "missing_percent": [dataframe[column].isna().mean() * 100 for column in dataframe.columns],
        "unique_values": [dataframe[column].nunique(dropna=True) for column in dataframe.columns],
    })

    summary_df["missing_percent"] = summary_df["missing_percent"].round(2)
    return summary_df


def count_duplicate_titles(dataframe, title_column="title"):
    """Count duplicate cocktail titles if the title column exists."""
    if title_column not in dataframe.columns:
        return None

    return dataframe[title_column].duplicated().sum()


def show_top_values(dataframe, column_name, top_n=10):
    """Show the most common values in one column."""
    if column_name not in dataframe.columns:
        print(f"Column not found: {column_name}")
        return

    top_values = dataframe[column_name].value_counts(dropna=False).head(top_n)
    display(top_values.to_frame(name="count"))


def calculate_text_lengths(dataframe, text_columns):
    """Calculate simple text length statistics for selected columns."""
    length_rows = []

    for column_name in text_columns:
        if column_name not in dataframe.columns:
            continue

        text_lengths = dataframe[column_name].fillna("").astype(str).str.len()

        length_rows.append({
            "column_name": column_name,
            "min_length": int(text_lengths.min()),
            "average_length": round(float(text_lengths.mean()), 2),
            "max_length": int(text_lengths.max()),
        })

    return pd.DataFrame(length_rows)


def show_sample_recipes(dataframe, sample_size=3):
    """Display a few recipes so we can inspect the raw format."""
    columns_to_show = [
        column_name
        for column_name in ["title", "glass", "garnish", "recipe", "ingredients"]
        if column_name in dataframe.columns
    ]

    display(dataframe[columns_to_show].head(sample_size))


# Confirm the raw dataset exists from Section 2.
if "cocktails_raw_df" not in globals():
    raise NameError("Please run Section 2 first so cocktails_raw_df is available.")


# 1. Show the basic dataset size.
print("Dataset shape")
show_dataset_shape(cocktails_raw_df)


# 2. Summarize each column.
print("\nColumn summary")
column_summary_df = create_column_summary(cocktails_raw_df)
display(column_summary_df)


# 3. Check for duplicate cocktail titles.
duplicate_title_count = count_duplicate_titles(cocktails_raw_df)
print("\nDuplicate title count:", duplicate_title_count)


# 4. Display a few raw recipe examples.
print("\nSample raw recipes")
show_sample_recipes(cocktails_raw_df, sample_size=3)


# 5. Show the most common glass types.
print("\nTop glass types")
show_top_values(cocktails_raw_df, column_name="glass", top_n=10)


# 6. Measure text lengths for fields that will later become recipe documents.
print("\nText length summary")
text_length_summary_df = calculate_text_lengths(
    cocktails_raw_df,
    text_columns=["title", "garnish", "recipe", "ingredients"],
)
display(text_length_summary_df)


Dataset shape
Rows: 6956
Columns: 5

Column summary


,column_name,data_type,missing_values,missing_percent,unique_values
0,title,object,0,0.00,6880
1,glass,object,0,0.00,46
2,garnish,object,289,4.15,2482
3,recipe,object,1,0.01,2973
4,ingredients,object,0,0.00,6928



Duplicate title count: 76

Sample raw recipes


,title,glass,garnish,recipe,ingredients
0,Abacaxi Ricaço,Pineapple shell (frozen) glass,Cut a straw sized hole in the top of the pinea...,Cut the top off a small pineapple and carefull...,"[['1 whole', 'Pineapple (fresh)'], ['9 cl', 'H..."
1,Abbey,Coupe glass,Orange zest twist,SHAKE all ingredients with ice and fine strain...,"[['4.5 cl', 'Rutte Dry Gin'], ['2.25 cl', 'Lil..."
2,A.B.C. Cocktail,Nick & Nora glass,Lemon zest twist & Luxardo Maraschino cherry,TEAR mint and place in shaker. Add other ingre...,"[['7 fresh', 'Mint leaves'], ['3 cl', 'Tawny p..."



Top glass types


,count
glass,
Coupe glass,2209
Martini glass,1131
Old-fashioned glass,1092
Collins glass,886
Nick & Nora glass,318
Highball (max 10oz/300ml),203
Rocks glass,146
Flute glass,143
Shot glass,136



Text length summary


,column_name,min_length,average_length,max_length
0,title,3,16.28,63
1,garnish,0,25.24,669
2,recipe,0,103.86,1278
3,ingredients,27,189.76,509


## Example Output

Your exact numbers may differ if the Hugging Face dataset is updated.

```text
Dataset shape
Rows: 6960
Columns: 5

Column summary
```

Example column summary table:

| column_name | data_type | missing_values | missing_percent | unique_values |
|---|---:|---:|---:|---:|
| title | object | 0 | 0.00 | 6960 |
| glass | object | 0 | 0.00 | many |
| garnish | object | some | varies | many |
| recipe | object | 0 | 0.00 | many |
| ingredients | object | 0 | 0.00 | many |

Example duplicate check:

```text
Duplicate title count: 0
```

Colab will also display sample recipe rows, common glass types, and text length statistics as tables.


## Short Comments

- We check shape first to understand the size of the project data.
- We inspect missing values because empty recipe fields can break matching and retrieval later.
- We check duplicate titles because duplicate recipes can bias recommendations.
- We inspect raw ingredient strings because Section 4 will need to parse them into clean ingredient lists.
- We measure text length because recipe, garnish, and ingredient fields will later become searchable documents.


# Section 4: Clean Dataset

In this section, we turn the raw cocktail data into a cleaner structure that later sections can use.

The most important cleaning task is parsing the `ingredients` column. In the Hugging Face dataset, each row stores ingredients as text that looks like a Python list of pairs:

```text
[['4.5 cl', 'Rutte Dry Gin'], ['2.25 cl', 'Lime juice (freshly squeezed)']]
```

We need to convert that text into real Python lists, separate the amount from the ingredient name, and create careful ingredient match keys.

This matters because not every `juice` is interchangeable. For example:

- `Passion fruit juice` should not match `lime juice` just because both contain the word `juice`.
- `Lemon juice (freshly squeezed)` should be treated as needing a fresh lemon, not bottled lemon juice.
- `Apple juice` should remain a fruit juice product.

## Why This Step Is Necessary

The raw dataset is useful for humans, but recommender systems need structured data.

Cleaning gives us:

- one recipe per row
- reliable text fields
- parsed ingredients
- display names for missing ingredients
- careful match keys for ingredient comparison
- a stable `recipe_id`

Later sections will use this cleaned data for ingredient matching, shopping lists, vector search, and recommendation prompts.

In [4]:
# ============================================================
# Section 4: Clean Dataset
# ============================================================

# Import libraries for parsing text safely and normalizing words.
import ast
import re
import unicodedata

import pandas as pd


FRESH_PREPARATION_MARKERS = [
    "freshly squeezed",
    "fresh squeezed",
    "fresh pressed",
    "fresh press",
    "freshly pressed",
]

FRESH_FRUIT_WORDS = {
    "lemon", "lime", "orange", "grapefruit", "pineapple", "passion fruit",
    "apple", "cranberry", "tomato", "pomegranate", "watermelon",
}

FRUIT_DETAIL_WORDS = {
    "fresh", "freshly", "squeezed", "pressed", "sweetened", "unsweetened",
    "pink", "white", "red", "green", "chilled",
}

SPIRIT_AND_COMMON_ALIASES = {
    "gin": ["gin", "dry gin", "old tom gin", "sloe gin", "genever"],
    "rum": ["rum", "light rum", "white rum", "dark rum", "aged rum", "navy rum", "jamaican rum"],
    "vodka": ["vodka"],
    "tequila": ["tequila"],
    "mezcal": ["mezcal"],
    "whiskey": ["whiskey", "whisky", "bourbon", "scotch", "rye whiskey", "rye whisky"],
    "brandy": ["brandy", "cognac", "calvados", "armagnac"],
    "absinthe": ["absinthe"],
    "pisco": ["pisco"],
    "cachaca": ["cachaca"],
    "sugar syrup": ["sugar syrup", "simple syrup", "rich syrup"],
    "honey syrup": ["honey syrup"],
    "grenadine": ["grenadine", "grenadine syrup"],
    "mint leaves": ["mint", "mint leaves"],
    "soda water": ["soda water", "club soda", "carbonated water"],
    "angostura bitters": ["angostura bitters", "angostura aromatic bitters"],
    "orange bitters": ["orange bitters"],
    "triple sec": ["triple sec"],
    "orange curacao liqueur": ["orange curacao", "orange curacao liqueur", "blue curacao"],
}


def clean_text(value):
    """Convert a value into clean plain text."""
    if value is None:
        return ""

    if not isinstance(value, (list, tuple, dict)) and pd.isna(value):
        return ""

    cleaned_value = str(value).strip()
    cleaned_value = re.sub(r"\s+", " ", cleaned_value)
    return cleaned_value


def remove_accents(text):
    """Convert accented characters into simpler ASCII-like text."""
    normalized_text = unicodedata.normalize("NFKD", text)
    ascii_text = normalized_text.encode("ascii", "ignore").decode("ascii")
    return ascii_text


def normalize_ingredient_name(ingredient_name):
    """Normalize an ingredient name for matching and grouping."""
    ingredient_text = clean_text(ingredient_name).lower()
    ingredient_text = remove_accents(ingredient_text)

    # Remove parenthetical details such as "(freshly squeezed)".
    ingredient_text = re.sub(r"\([^)]*\)", " ", ingredient_text)

    # Replace punctuation with spaces, then compress extra spaces.
    ingredient_text = re.sub(r"[^a-z0-9]+", " ", ingredient_text)
    ingredient_text = re.sub(r"\s+", " ", ingredient_text).strip()
    return ingredient_text


def normalize_with_details(ingredient_name):
    """Normalize text while keeping useful parenthetical words such as freshly squeezed."""
    ingredient_text = clean_text(ingredient_name).lower()
    ingredient_text = remove_accents(ingredient_text)
    ingredient_text = re.sub(r"[^a-z0-9]+", " ", ingredient_text)
    ingredient_text = re.sub(r"\s+", " ", ingredient_text).strip()
    return ingredient_text


def contains_whole_phrase(text, phrase):
    """Check whether a phrase appears as a whole phrase inside normalized text."""
    return bool(re.search(rf"\b{re.escape(phrase)}\b", text))


def has_fresh_preparation_marker(raw_ingredient_name):
    """Detect whether a juice ingredient explicitly asks for fresh preparation."""
    normalized_with_details = normalize_with_details(raw_ingredient_name)
    return any(marker in normalized_with_details for marker in FRESH_PREPARATION_MARKERS)


def extract_fruit_before_juice(normalized_without_details):
    """Extract the fruit phrase before the word juice."""
    juice_match = re.search(r"\bjuice\b", normalized_without_details)

    if not juice_match:
        return None

    fruit_part = normalized_without_details[:juice_match.start()].strip()
    fruit_tokens = [
        token
        for token in fruit_part.split()
        if token not in FRUIT_DETAIL_WORDS
    ]
    fruit_name = " ".join(fruit_tokens).strip()
    return fruit_name or None


def canonicalize_alias_ingredient(normalized_without_details):
    """Map common broad ingredient names to stable keys without overmatching juices."""
    for canonical_name, aliases in SPIRIT_AND_COMMON_ALIASES.items():
        for alias in aliases:
            if contains_whole_phrase(normalized_without_details, alias):
                return canonical_name

    return None


def create_ingredient_match_key(ingredient_name):
    """Create a careful match key for one ingredient.

    This function is stricter than token overlap. It keeps fruit juices distinct
    and treats explicitly fresh citrus juice as fresh fruit.
    """
    normalized_without_details = normalize_ingredient_name(ingredient_name)

    if not normalized_without_details:
        return ""

    fruit_before_juice = extract_fruit_before_juice(normalized_without_details)

    if fruit_before_juice:
        if has_fresh_preparation_marker(ingredient_name):
            return f"fresh {fruit_before_juice}"
        return f"{fruit_before_juice} juice"

    if normalized_without_details in FRESH_FRUIT_WORDS:
        return f"fresh {normalized_without_details}"

    if normalized_without_details.startswith("fresh "):
        possible_fruit = normalized_without_details.replace("fresh ", "", 1).strip()
        if possible_fruit in FRESH_FRUIT_WORDS:
            return f"fresh {possible_fruit}"

    alias_key = canonicalize_alias_ingredient(normalized_without_details)

    if alias_key:
        return alias_key

    return normalized_without_details


def create_display_ingredient_name(ingredient_name):
    """Create a human-readable ingredient name for matching output and shopping lists."""
    normalized_without_details = normalize_ingredient_name(ingredient_name)
    fruit_before_juice = extract_fruit_before_juice(normalized_without_details)

    if fruit_before_juice and has_fresh_preparation_marker(ingredient_name):
        return f"fresh {fruit_before_juice} (for fresh juice)"

    return clean_text(ingredient_name)


def parse_ingredients_value(ingredients_value):
    """Parse one raw ingredients value into a list of dictionaries."""
    if ingredients_value is None:
        return []

    if isinstance(ingredients_value, list):
        ingredient_pairs = ingredients_value
    else:
        if pd.isna(ingredients_value) or str(ingredients_value).strip() == "":
            return []

        try:
            ingredient_pairs = ast.literal_eval(str(ingredients_value))
        except (ValueError, SyntaxError):
            ingredient_pairs = []

    parsed_ingredients = []

    for ingredient_pair in ingredient_pairs:
        if not isinstance(ingredient_pair, (list, tuple)) or len(ingredient_pair) < 2:
            continue

        amount_text = clean_text(ingredient_pair[0])
        ingredient_text = clean_text(ingredient_pair[1])
        normalized_name = normalize_ingredient_name(ingredient_text)
        match_key = create_ingredient_match_key(ingredient_text)
        display_name = create_display_ingredient_name(ingredient_text)

        if match_key:
            parsed_ingredients.append({
                "amount": amount_text,
                "ingredient": ingredient_text,
                "normalized_ingredient": normalized_name,
                "match_key": match_key,
                "display_ingredient": display_name,
            })

    return parsed_ingredients


def extract_unique_field(parsed_ingredients, field_name):
    """Extract one field from parsed ingredient dictionaries while removing duplicates."""
    values = [
        ingredient[field_name]
        for ingredient in parsed_ingredients
        if ingredient.get(field_name)
    ]
    return list(dict.fromkeys(values))


def build_search_text(row):
    """Create a readable text field that combines important recipe information."""
    ingredient_text = ", ".join(row["ingredient_display_names"])
    text_parts = [
        f"Title: {row['title']}",
        f"Glass: {row['glass']}",
        f"Garnish: {row['garnish']}",
        f"Ingredients: {ingredient_text}",
        f"Instructions: {row['recipe']}",
    ]
    return "\n".join(text_parts)


def clean_cocktail_dataframe(raw_dataframe):
    """Clean the raw cocktail recipe DataFrame."""
    cleaned_dataframe = raw_dataframe.copy()

    # Clean text columns without changing the original DataFrame.
    for column_name in ["title", "glass", "garnish", "recipe", "ingredients"]:
        if column_name in cleaned_dataframe.columns:
            cleaned_dataframe[column_name] = cleaned_dataframe[column_name].apply(clean_text)

    # Keep rows that have the minimum information needed for recommendation.
    cleaned_dataframe = cleaned_dataframe[
        (cleaned_dataframe["title"] != "")
        & (cleaned_dataframe["recipe"] != "")
        & (cleaned_dataframe["ingredients"] != "")
    ].copy()

    # Parse ingredients into structured lists.
    cleaned_dataframe["parsed_ingredients"] = cleaned_dataframe["ingredients"].apply(parse_ingredients_value)
    cleaned_dataframe["ingredient_names"] = cleaned_dataframe["parsed_ingredients"].apply(
        lambda ingredients: extract_unique_field(ingredients, "normalized_ingredient")
    )
    cleaned_dataframe["ingredient_match_keys"] = cleaned_dataframe["parsed_ingredients"].apply(
        lambda ingredients: extract_unique_field(ingredients, "match_key")
    )
    cleaned_dataframe["ingredient_display_names"] = cleaned_dataframe["parsed_ingredients"].apply(
        lambda ingredients: extract_unique_field(ingredients, "display_ingredient")
    )
    cleaned_dataframe["ingredient_count"] = cleaned_dataframe["ingredient_match_keys"].apply(len)

    # Remove rows where ingredients could not be parsed.
    cleaned_dataframe = cleaned_dataframe[cleaned_dataframe["ingredient_count"] > 0].copy()

    # Remove duplicate cocktail titles while keeping the first version.
    cleaned_dataframe = cleaned_dataframe.drop_duplicates(subset="title", keep="first")

    # Add a simple stable ID for later lookup.
    cleaned_dataframe = cleaned_dataframe.reset_index(drop=True)
    cleaned_dataframe["recipe_id"] = cleaned_dataframe.index

    # Build reusable recipe text for future document creation.
    cleaned_dataframe["search_text"] = cleaned_dataframe.apply(build_search_text, axis=1)

    return cleaned_dataframe


# Confirm the raw dataset exists from Section 2.
if "cocktails_raw_df" not in globals():
    raise NameError("Please run Section 2 first so cocktails_raw_df is available.")


# Create the cleaned dataset used by all later sections.
cocktails_clean_df = clean_cocktail_dataframe(cocktails_raw_df)

# Show what changed after cleaning.
print("Raw dataset shape:", cocktails_raw_df.shape)
print("Cleaned dataset shape:", cocktails_clean_df.shape)
print("Rows removed:", len(cocktails_raw_df) - len(cocktails_clean_df))

print("\nCleaned columns:")
print(list(cocktails_clean_df.columns))

print("\nSample cleaned ingredient data:")
display(cocktails_clean_df[[
    "recipe_id",
    "title",
    "ingredient_display_names",
    "ingredient_match_keys",
    "ingredient_count",
]].head(5))

Raw dataset shape: (6956, 5)
Cleaned dataset shape: (6879, 12)
Rows removed: 77

Cleaned columns:
['title', 'glass', 'garnish', 'recipe', 'ingredients', 'parsed_ingredients', 'ingredient_names', 'ingredient_match_keys', 'ingredient_display_names', 'ingredient_count', 'recipe_id', 'search_text']

Sample cleaned ingredient data:


,recipe_id,title,ingredient_display_names,ingredient_match_keys,ingredient_count
0,0,Abacaxi Ricaço,"[Pineapple (fresh), Havana Club 3 Year Old rum...","[fresh pineapple, rum, fresh lime, white caste...",4
1,1,Abbey,"[Rutte Dry Gin, Lillet Blanc (or other aromati...","[gin, lillet blanc, fresh orange, angostura bi...",4
2,2,A.B.C. Cocktail,"[Mint leaves, Tawny port, Rémy Martin 1738 Cog...","[mint leaves, tawny port, brandy, luxardo mara...",5
3,3,Absinthe Cocktail,"[La Fée Parisienne absinthe, Chilled water, Su...","[absinthe, chilled water, sugar syrup]",3
4,4,Absinthe Frappé,"[La Fée Parisienne absinthe, Anisette liqueur,...","[absinthe, anisette liqueur, chilled water, su...",4


## Example Output

Your exact numbers may differ if the dataset is updated.

```text
Raw dataset shape: (6960, 5)
Cleaned dataset shape: (6960, 12)
Rows removed: 0

Cleaned columns:
['title', 'glass', 'garnish', 'recipe', 'ingredients', 'parsed_ingredients', 'ingredient_names', 'ingredient_match_keys', 'ingredient_display_names', 'ingredient_count', 'recipe_id', 'search_text']
```

Example cleaned ingredient row:

| recipe_id | title | ingredient_display_names | ingredient_match_keys | ingredient_count |
|---:|---|---|---|---:|
| 0 | Abacaxi Ricaco | ['Pineapple (fresh)', 'Havana Club 3 Year Old rum', 'fresh lime (for fresh juice)', 'White caster sugar'] | ['fresh pineapple', 'rum', 'fresh lime', 'white caster sugar'] | 4 |

Notice that `passion fruit juice`, `apple juice`, and `fresh lime` are separate match keys.

## Short Comments

- We keep `cocktails_raw_df` unchanged so we can always return to the original data.
- We create `cocktails_clean_df` as the reliable working dataset.
- We parse ingredients into dictionaries because later functions need ingredient names separately from measurements.
- We create `ingredient_match_keys` so broad words like `juice` do not cause false matches.
- We treat explicitly fresh juice requirements as fresh fruit, such as `fresh lemon`, instead of bottled juice.

# Section 5: Ingredient Matching

This section builds the custom Ingredient Matching Engine.

Given a list of ingredients a user has at home, the engine calculates for every cocktail:

- matched ingredients
- missing ingredients
- match percentage

The matching now uses careful ingredient keys from Section 4. This prevents false matches such as `passion fruit juice` matching `lime juice`, and it keeps `fresh lemon` separate from bottled `lemon juice`.

## Why This Step Is Necessary

A home mixologist should not recommend drinks only because they sound similar to a query.

It should also answer a practical question:

```text
What can I make with what I already have?
```

This matching engine gives the project a rule-based recommendation layer before we add vector search and LLM prompting.

Careful ingredient identity is important. In cocktails, `apple juice`, `passion fruit juice`, `fresh lemon`, and `orange cura?ao liqueur` are different materials even if some of their words overlap.

In [5]:
# ============================================================
# Section 5: Ingredient Matching
# ============================================================

import pandas as pd


def parse_available_ingredients(available_ingredients):
    """Convert user-provided ingredients into display names and strict match keys."""
    parsed_available_ingredients = []

    for available_ingredient in available_ingredients:
        ingredient_text = clean_text(available_ingredient)
        ingredient_key = create_ingredient_match_key(ingredient_text)

        if ingredient_key:
            parsed_available_ingredients.append({
                "user_text": ingredient_text,
                "match_key": ingredient_key,
            })

    return parsed_available_ingredients


def ingredient_key_is_available(recipe_match_key, parsed_available_ingredients):
    """Check whether a recipe ingredient key exists in the user's available keys."""
    available_keys = {
        ingredient["match_key"]
        for ingredient in parsed_available_ingredients
    }
    return recipe_match_key in available_keys


def get_recipe_ingredient_items(cocktail_row):
    """Return unique recipe ingredient items with both display names and match keys."""
    recipe_items = []
    seen_match_keys = set()

    for ingredient in cocktail_row["parsed_ingredients"]:
        match_key = ingredient.get("match_key", "")
        display_name = ingredient.get("display_ingredient", ingredient.get("ingredient", ""))

        if match_key and match_key not in seen_match_keys:
            recipe_items.append({
                "display_ingredient": display_name,
                "match_key": match_key,
            })
            seen_match_keys.add(match_key)

    return recipe_items


def match_single_cocktail(cocktail_row, available_ingredients):
    """Calculate ingredient matching details for one cocktail."""
    parsed_available_ingredients = parse_available_ingredients(available_ingredients)
    recipe_items = get_recipe_ingredient_items(cocktail_row)

    matched_ingredients = []
    missing_ingredients = []
    matched_ingredient_keys = []
    missing_ingredient_keys = []

    for recipe_item in recipe_items:
        recipe_key = recipe_item["match_key"]
        display_name = recipe_item["display_ingredient"]

        if ingredient_key_is_available(recipe_key, parsed_available_ingredients):
            matched_ingredients.append(display_name)
            matched_ingredient_keys.append(recipe_key)
        else:
            missing_ingredients.append(display_name)
            missing_ingredient_keys.append(recipe_key)

    total_ingredients = len(recipe_items)
    matched_count = len(matched_ingredients)
    missing_count = len(missing_ingredients)

    if total_ingredients == 0:
        match_percentage = 0.0
    else:
        match_percentage = round((matched_count / total_ingredients) * 100, 2)

    return {
        "recipe_id": cocktail_row["recipe_id"],
        "title": cocktail_row["title"],
        "glass": cocktail_row["glass"],
        "total_ingredients": total_ingredients,
        "matched_count": matched_count,
        "missing_count": missing_count,
        "match_percentage": match_percentage,
        "matched_ingredients": matched_ingredients,
        "missing_ingredients": missing_ingredients,
        "matched_ingredient_keys": matched_ingredient_keys,
        "missing_ingredient_keys": missing_ingredient_keys,
    }


def rank_cocktails_by_ingredients(clean_dataframe, available_ingredients, top_n=10):
    """Rank cocktails by how well they match the user's available ingredients."""
    match_rows = []

    for _, cocktail_row in clean_dataframe.iterrows():
        match_rows.append(match_single_cocktail(cocktail_row, available_ingredients))

    match_results_df = pd.DataFrame(match_rows)
    match_results_df = match_results_df.sort_values(
        by=["match_percentage", "missing_count", "matched_count", "title"],
        ascending=[False, True, False, True],
    ).reset_index(drop=True)

    if top_n is None:
        return match_results_df

    return match_results_df.head(top_n)


# Confirm the cleaned dataset exists from Section 4.
if "cocktails_clean_df" not in globals():
    raise NameError("Please run Section 4 first so cocktails_clean_df is available.")


# Example home bar ingredients for the teaching demo.
# Fresh citrus is written as fresh fruit because recipes such as
# "Lemon juice (freshly squeezed)" require fresh lemon, not bottled juice.
user_available_ingredients = [
    "gin",
    "rum",
    "vodka",
    "fresh lime",
    "fresh lemon",
    "orange juice",
    "sugar syrup",
    "mint leaves",
    "soda water",
    "angostura bitters",
]

# Rank the top cocktail matches for this example home bar.
matching_results_df = rank_cocktails_by_ingredients(
    cocktails_clean_df,
    user_available_ingredients,
    top_n=10,
)

print("Available ingredients:")
print(user_available_ingredients)

print("\nAvailable ingredient match keys:")
print([ingredient["match_key"] for ingredient in parse_available_ingredients(user_available_ingredients)])

print("\nTop matching cocktails:")
display(matching_results_df[[
    "title",
    "match_percentage",
    "matched_count",
    "missing_count",
    "matched_ingredients",
    "missing_ingredients",
]])

Available ingredients:
['gin', 'rum', 'vodka', 'fresh lime', 'fresh lemon', 'orange juice', 'sugar syrup', 'mint leaves', 'soda water', 'angostura bitters']

Available ingredient match keys:
['gin', 'rum', 'vodka', 'fresh lime', 'fresh lemon', 'orange juice', 'sugar syrup', 'mint leaves', 'soda water', 'angostura bitters']

Top matching cocktails:


,title,match_percentage,matched_count,missing_count,matched_ingredients,missing_ingredients
0,Ray's Hard Lemonade,100.0,6,0,"[Mint leaves, Ketel One Vodka, fresh lemon (fo...",[]
1,Angela's Mojito Cocktail,100.0,5,0,"[Mint leaves, fresh lime (for fresh juice), Su...",[]
2,Major Bailey #1,100.0,5,0,"[Mint leaves, Rutte Dry Gin, fresh lime (for f...",[]
3,Mint Collins,100.0,5,0,"[Mint leaves, Rutte Dry Gin, fresh lemon (for ...",[]
4,Mojito Cocktail,100.0,5,0,"[Mint leaves, Havana Club 3 Year Old rum, fres...",[]
5,Mojito Through The Straw,100.0,5,0,"[Mint leaves, 'Simple' sugar syrup (1 sugar to...",[]
6,Momo Special,100.0,5,0,"[Mint leaves, Ketel One Vodka, fresh lime (for...",[]
7,Olle's Gin Mint Smash,100.0,5,0,"[Gin, Mint leaves, Sugar syrup (65.0°brix, 2 s...",[]
8,Queen's Park Swizzle,100.0,5,0,"[Mint leaves, Havana Club 3 Year Old rum, fres...",[]
9,South Side Rickey,100.0,5,0,"[Rutte Dry Gin, fresh lime (for fresh juice), ...",[]


## Example Output

```text
Available ingredients:
['gin', 'rum', 'vodka', 'fresh lime', 'fresh lemon', 'orange juice', 'sugar syrup', 'mint leaves', 'soda water', 'angostura bitters']

Available ingredient match keys:
['gin', 'rum', 'vodka', 'fresh lime', 'fresh lemon', 'orange juice', 'sugar syrup', 'mint leaves', 'soda water', 'angostura bitters']
```

Example recommendation table:

| title | match_percentage | matched_count | missing_count | missing_ingredients |
|---|---:|---:|---:|---|
| Example Cocktail A | 100.00 | 3 | 0 | [] |
| Example Cocktail B | 80.00 | 4 | 1 | ['passion fruit juice'] |
| Example Cocktail C | 75.00 | 3 | 1 | ['orange cura?ao liqueur'] |

The exact cocktail names depend on the current dataset content.

The important behavior is that `passion fruit juice` is no longer treated as available just because the user has another kind of juice.

## Short Comments

- We compare strict ingredient match keys instead of loose word overlap.
- We calculate both matched and missing ingredients so recommendations are explainable.
- We rank by `match_percentage` first because the easiest cocktails should appear first.
- We distinguish fresh citrus requirements from bottled juices.
- This is a custom matching engine, not a LangChain or framework-based component.

# Section 6: Shopping List Generator

This section builds the custom Shopping List Generator.

The goal is to look at recommended cocktails, collect their missing ingredients, merge duplicates, and count how frequently each missing ingredient appears.

## Why This Step Is Necessary

Recommendations are more useful when they lead to action.

If the system recommends several cocktails but each one is missing ingredients, the user needs a clear shopping list.

Counting ingredient frequency helps the user see which purchases support the most recommended drinks.

In [6]:
# ============================================================
# Section 6: Shopping List Generator
# ============================================================

from collections import Counter

import pandas as pd


def choose_shopping_label(ingredient_key, display_ingredient):
    """Choose a clean shopping label for one missing ingredient."""
    if ingredient_key.startswith("fresh "):
        return ingredient_key
    return ingredient_key or display_ingredient


def generate_shopping_list(recommendations_df, top_n=None):
    """Generate a merged shopping list from cocktail recommendation results."""
    if top_n is not None:
        recommendations_to_use = recommendations_df.head(top_n)
    else:
        recommendations_to_use = recommendations_df

    missing_ingredient_counter = Counter()
    cocktail_titles_by_ingredient = {}
    display_label_by_ingredient = {}

    for _, recommendation_row in recommendations_to_use.iterrows():
        cocktail_title = recommendation_row["title"]
        missing_ingredients = recommendation_row["missing_ingredients"]
        missing_ingredient_keys = recommendation_row.get("missing_ingredient_keys", missing_ingredients)

        for ingredient_key, display_ingredient in zip(missing_ingredient_keys, missing_ingredients):
            shopping_label = choose_shopping_label(ingredient_key, display_ingredient)
            missing_ingredient_counter[shopping_label] += 1
            cocktail_titles_by_ingredient.setdefault(shopping_label, []).append(cocktail_title)
            display_label_by_ingredient.setdefault(shopping_label, display_ingredient)

    shopping_rows = []

    for ingredient_name, frequency in missing_ingredient_counter.items():
        shopping_rows.append({
            "ingredient": ingredient_name,
            "example_dataset_label": display_label_by_ingredient[ingredient_name],
            "needed_for_cocktail_count": frequency,
            "example_cocktails": cocktail_titles_by_ingredient[ingredient_name][:5],
        })

    shopping_list_df = pd.DataFrame(shopping_rows)

    if shopping_list_df.empty:
        return pd.DataFrame(columns=[
            "ingredient",
            "example_dataset_label",
            "needed_for_cocktail_count",
            "example_cocktails",
        ])

    shopping_list_df = shopping_list_df.sort_values(
        by=["needed_for_cocktail_count", "ingredient"],
        ascending=[False, True],
    ).reset_index(drop=True)

    return shopping_list_df


# Confirm matching results exist from Section 5.
if "matching_results_df" not in globals():
    raise NameError("Please run Section 5 first so matching_results_df is available.")


# Build a shopping list from the top 10 recommended cocktails.
shopping_list_df = generate_shopping_list(matching_results_df, top_n=10)

print("Shopping list from the top 10 recommended cocktails:")
display(shopping_list_df)

Shopping list from the top 10 recommended cocktails:


,ingredient,example_dataset_label,needed_for_cocktail_count,example_cocktails


## Example Output

Example shopping list:

| ingredient | example_dataset_label | needed_for_cocktail_count | example_cocktails |
|---|---|---:|---|
| passion fruit juice | Passion fruit juice | 3 | ['Cocktail A', 'Cocktail B', 'Cocktail C'] |
| fresh lemon | fresh lemon (for fresh juice) | 2 | ['Cocktail D', 'Cocktail E'] |
| orange curacao liqueur | Orange Cura?ao liqueur | 1 | ['Cocktail F'] |

If several top recommendations are already fully makeable, the shopping list may be short.

## Short Comments

- We merge duplicate missing ingredients using the same strict match keys from Section 5.
- We count frequency because a frequently missing ingredient may be a smart purchase.
- We keep example cocktail titles so the shopping list remains explainable.
- We keep an example dataset label so users can see how the ingredient appeared in a recipe.
- This section uses the output of the custom matching engine from Section 5.

# Section 7: Best Value Purchase

This section builds the custom Best Value Purchase Analyzer.

We simulate buying exactly one additional ingredient and ask:

```text
How many new cocktails would become fully available?
```

Then we rank ingredients by the number of recipes they unlock.

## Why This Step Is Necessary

A shopping list tells the user what is missing.

A best-value analyzer goes one step further and helps the user decide what to buy first.

This is useful for a home bar because one good ingredient can unlock many new recipes.

In [7]:
# ============================================================
# Section 7: Best Value Purchase
# ============================================================

from collections import defaultdict

import pandas as pd


def get_fully_available_count(match_results_df):
    """Count cocktails where no ingredients are missing."""
    return int((match_results_df["missing_count"] == 0).sum())


def analyze_best_value_purchase(clean_dataframe, available_ingredients, top_n=10):
    """Rank one-ingredient purchases by how many cocktails they unlock."""
    all_matches_df = rank_cocktails_by_ingredients(
        clean_dataframe,
        available_ingredients,
        top_n=None,
    )

    baseline_available_count = get_fully_available_count(all_matches_df)
    unlocked_titles_by_ingredient = defaultdict(list)
    display_label_by_ingredient = {}

    # If a cocktail has exactly one missing ingredient, buying that ingredient unlocks it.
    one_missing_df = all_matches_df[all_matches_df["missing_count"] == 1]

    for _, match_row in one_missing_df.iterrows():
        missing_ingredient_key = match_row["missing_ingredient_keys"][0]
        missing_ingredient_display = match_row["missing_ingredients"][0]
        purchase_label = choose_shopping_label(missing_ingredient_key, missing_ingredient_display)

        unlocked_titles_by_ingredient[purchase_label].append(match_row["title"])
        display_label_by_ingredient.setdefault(purchase_label, missing_ingredient_display)

    value_rows = []

    for ingredient_name, unlocked_titles in unlocked_titles_by_ingredient.items():
        unlocked_count = len(unlocked_titles)
        value_rows.append({
            "ingredient_to_buy": ingredient_name,
            "example_dataset_label": display_label_by_ingredient[ingredient_name],
            "currently_makeable_cocktails": baseline_available_count,
            "extra_cocktails_unlocked": unlocked_count,
            "total_makeable_after_purchase": baseline_available_count + unlocked_count,
            "example_unlocked_cocktails": unlocked_titles[:5],
        })

    best_value_df = pd.DataFrame(value_rows)

    if best_value_df.empty:
        return pd.DataFrame(columns=[
            "ingredient_to_buy",
            "example_dataset_label",
            "currently_makeable_cocktails",
            "extra_cocktails_unlocked",
            "total_makeable_after_purchase",
            "example_unlocked_cocktails",
        ])

    best_value_df = best_value_df.sort_values(
        by=["extra_cocktails_unlocked", "ingredient_to_buy"],
        ascending=[False, True],
    ).head(top_n).reset_index(drop=True)

    return best_value_df


# Confirm the cleaned dataset and example ingredients exist.
if "cocktails_clean_df" not in globals() or "user_available_ingredients" not in globals():
    raise NameError("Please run Sections 4 and 5 first.")


# Find the top 10 best one-ingredient purchases.
best_value_purchase_df = analyze_best_value_purchase(
    cocktails_clean_df,
    user_available_ingredients,
    top_n=10,
)

print("Top 10 best value ingredients to buy next:")
display(best_value_purchase_df)

Top 10 best value ingredients to buy next:


,ingredient_to_buy,example_dataset_label,currently_makeable_cocktails,extra_cocktails_unlocked,total_makeable_after_purchase,example_unlocked_cocktails
0,brandy,Cherry Heering cherry brandy liqueur,69,25,94,"[Cherry Mojito, Fog Cutter (Bramble style), Sl..."
1,whiskey,Bourbon whiskey,69,22,91,"[Jack Tar, The Modern Whisky Sour, Whiskey Col..."
2,chilled water,Chilled water,69,20,89,"[Francis's Victorian Lemonade, Gin Punch #1, G..."
3,triple sec,De Kuyper Triple Sec (40%),69,20,89,"[Fine & Dandy, Tears of Joy, Angela's Lemon Dr..."
4,fresh pineapple,fresh pineapple (for fresh juice),69,18,87,"[Blade Runner, Pineapple Fizz, Tabu, Hunk Cock..."
5,martini extra dry vermouth,Martini Extra Dry vermouth,69,16,85,"[Cameo Kirby, Cuban Island, President Vincent,..."
6,pasteurised egg white,Pasteurised egg white,69,13,82,"[Amado, Gin Sour, Sour (Generic Name), Vodka S..."
7,grapefruit juice,Grapefruit juice (pink),69,12,81,"[Gin Paloma, Swedish Paloma, Black Daiquiri, G..."
8,grenadine,Monin Grenadine syrup,69,10,79,"[Happy Together, Shark's Tooth No.1 (Trader Vi..."
9,thomas henry ginger beer,Thomas Henry Ginger Beer,69,10,79,"[Dark 'N' Stormy (Difford's recipe), Dutch Mul..."


## Example Output

Example best-value table:

| ingredient_to_buy | example_dataset_label | currently_makeable_cocktails | extra_cocktails_unlocked | total_makeable_after_purchase | example_unlocked_cocktails |
|---|---|---:|---:|---:|---|
| passion fruit juice | Passion fruit juice | 12 | 18 | 30 | ['Cocktail A', 'Cocktail B'] |
| fresh lemon | fresh lemon (for fresh juice) | 12 | 14 | 26 | ['Cocktail C', 'Cocktail D'] |
| orange curacao liqueur | Orange Cura?ao liqueur | 12 | 9 | 21 | ['Cocktail E'] |

The exact ranking depends on the available ingredients list.

## Short Comments

- We treat a cocktail with `missing_count == 1` as unlockable by one purchase.
- We rank ingredients by how many recipes they unlock.
- We group purchases by strict match key, so different juices and different fresh fruits stay separate.
- We include example unlocked cocktail titles so the recommendation is easy to explain.
- This feature is custom logic built on top of the Ingredient Matching Engine.

# Section 8: Convert Recipes into Documents

This section converts each cleaned cocktail recipe into a text document.

In a manual RAG pipeline, a document is simply a searchable piece of text. Each cocktail becomes one document containing:

- title
- glass type
- garnish
- ingredients
- instructions

## Why This Step Is Necessary

Embedding models do not work directly with messy DataFrame rows.

They work best with readable text.

By converting each recipe into a document, we prepare the dataset for semantic search in later sections.

In [8]:
# ============================================================
# Section 8: Convert Recipes into Documents
# ============================================================

import pandas as pd


def format_ingredient_lines(parsed_ingredients):
    """Format parsed ingredients into readable document lines."""
    ingredient_lines = []

    for ingredient in parsed_ingredients:
        amount = ingredient.get("amount", "")
        ingredient_name = ingredient.get("ingredient", "")

        if amount:
            ingredient_lines.append(f"- {amount} {ingredient_name}")
        else:
            ingredient_lines.append(f"- {ingredient_name}")

    return "\n".join(ingredient_lines)


def create_recipe_document(cocktail_row):
    """Convert one cleaned cocktail row into a document dictionary."""
    ingredient_lines = format_ingredient_lines(cocktail_row["parsed_ingredients"])

    document_text = f"""
Cocktail: {cocktail_row['title']}
Glass: {cocktail_row['glass']}
Garnish: {cocktail_row['garnish']}

Ingredients:
{ingredient_lines}

Instructions:
{cocktail_row['recipe']}
""".strip()

    return {
        "document_id": int(cocktail_row["recipe_id"]),
        "title": cocktail_row["title"],
        "text": document_text,
        "ingredient_names": cocktail_row["ingredient_display_names"],
        "ingredient_match_keys": cocktail_row["ingredient_match_keys"],
    }


def create_recipe_documents(clean_dataframe):
    """Create one searchable document for every cocktail recipe."""
    recipe_documents = []

    for _, cocktail_row in clean_dataframe.iterrows():
        recipe_documents.append(create_recipe_document(cocktail_row))

    return recipe_documents


# Confirm cleaned data exists from Section 4.
if "cocktails_clean_df" not in globals():
    raise NameError("Please run Section 4 first so cocktails_clean_df is available.")


# Convert cleaned cocktail rows into plain text documents.
recipe_documents = create_recipe_documents(cocktails_clean_df)
recipe_documents_df = pd.DataFrame(recipe_documents)

print(f"Created {len(recipe_documents)} recipe documents.")
print("\nSample document:")
print(recipe_documents[0]["text"][:1000])

Created 6879 recipe documents.

Sample document:
Cocktail: Abacaxi Ricaço
Glass: Pineapple shell (frozen) glass
Garnish: Cut a straw sized hole in the top of the pineapple shell & replace it as a lid

Ingredients:
- 1 whole Pineapple (fresh)
- 9 cl Havana Club 3 Year Old rum
- 2.25 cl Lime juice (freshly squeezed)
- 1.5 cl White caster sugar

Instructions:
Cut the top off a small pineapple and carefully scoop out the flesh from the base to leave a shell with 12mm (½ inch) thick walls. Place the shell in a freezer to chill. Remove the hard core from the pineapple flesh and discard; roughly chop the remaining flesh, add other ingredients and BLEND with one 12oz scoop of crushed ice. Pour into the pineapple shell and serve with straws. (The flesh of one pineapple blended with the following ingredients will fill at least two shells).


## Example Output

```text
Created 6960 recipe documents.

Sample document:
Cocktail: Abacaxi Ricaco
Glass: Pineapple shell (frozen) glass
Garnish: Cut a straw sized hole in the top of the pineapple shell...

Ingredients:
- 1 whole Pineapple (fresh)
- 9 cl Havana Club 3 Year Old rum
...
```

## Short Comments

- We create one document per cocktail because each cocktail is a natural retrieval unit.
- We keep `document_id` aligned with `recipe_id` so search results can link back to the DataFrame.
- We include ingredient amounts in the document for better final recommendations.
- This is the document creation step of the manual RAG pipeline.

# Section 9: Generate Embeddings

This section turns recipe documents into embeddings.

An embedding is a list of numbers that represents the meaning of a piece of text. Similar texts should have similar embeddings.

Following the simpler RAG notebook style, we keep this section direct: load one embedding model, collect document texts, encode them, and store the result as a NumPy matrix.

## Why This Step Is Necessary

RAG systems need a way to find relevant documents.

Keyword search only matches exact words. Embedding search can match meaning.

For example, a query about "citrus gin drink" may retrieve recipes with fresh lemon, fresh lime, and gin even if the exact wording is different.

In [9]:
# ============================================================
# Section 9: Generate Embeddings
# ============================================================

import numpy as np
from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# Confirm recipe documents exist from Section 8.
if "recipe_documents" not in globals():
    raise NameError("Please run Section 8 first so recipe_documents is available.")


# 1. Load a compact sentence-transformers embedding model.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# 2. Collect the text from every recipe document.
document_texts = [document["text"] for document in recipe_documents]

# 3. Convert the document text into normalized embeddings.
document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# 4. FAISS expects float32 vectors.
document_embeddings = np.array(document_embeddings).astype("float32")

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Number of documents:", len(document_texts))
print("Embedding matrix shape:", document_embeddings.shape)
print("Embedding data type:", document_embeddings.dtype)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/215 [00:00<?, ?it/s]

Embedding model: all-MiniLM-L6-v2
Number of documents: 6879
Embedding matrix shape: (6879, 384)
Embedding data type: float32


## Example Output

```text
Embedding model: all-MiniLM-L6-v2
Number of documents: 6960
Embedding matrix shape: (6960, 384)
Embedding data type: float32
```

The first run may take longer because Colab needs to download the embedding model.

## Short Comments

- We embed the full recipe document, not just the cocktail title.
- We normalize embeddings so inner product search behaves like cosine similarity.
- We store embeddings in `float32` because FAISS expects efficient numeric arrays.
- This is the embeddings step of the manual RAG pipeline.

# Section 10: Create FAISS Index

This section creates a FAISS vector index.

FAISS stores document embeddings and lets us quickly search for the most similar documents to a user query.

## Why This Step Is Necessary

Embeddings are useful only if we can search them efficiently.

A vector index lets the notebook retrieve relevant cocktail documents quickly, even when there are thousands of recipes.

We use `IndexFlatIP` because our embeddings are normalized, so inner product acts like cosine similarity.

In [10]:
# ============================================================
# Section 10: Create FAISS Index
# ============================================================

import faiss

# Confirm embeddings exist from Section 9.
if "document_embeddings" not in globals():
    raise NameError("Please run Section 9 first so document_embeddings is available.")


# The embedding dimension is the number of values in each vector.
embedding_dimension = document_embeddings.shape[1]

# Because embeddings are normalized, inner product works like cosine similarity.
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(document_embeddings)

print("FAISS index created.")
print("Number of vectors in index:", faiss_index.ntotal)
print("Embedding dimension:", embedding_dimension)

FAISS index created.
Number of vectors in index: 6879
Embedding dimension: 384


## Example Output

```text
FAISS index created.
Number of vectors in index: 6960
Embedding dimension: 384
```

## Short Comments

- FAISS stores the embedding vectors for fast similarity search.
- `IndexFlatIP` is simple and beginner-friendly because it searches all vectors exactly.
- We use normalized embeddings, so higher scores mean more similar documents.
- This is the vector search index step of the manual RAG pipeline.

# Section 11: RAG Retrieval

This section performs the retrieval part of RAG manually.

The retrieval process is:

1. Convert the user query into an embedding.
2. Search the FAISS index.
3. Return the most similar cocktail documents.
4. Format those documents as context for the LLM.

## Why This Step Is Necessary

An LLM should not recommend from memory alone.

Retrieval gives the LLM grounded recipe information from our cocktail dataset.

This makes recommendations more relevant and easier to explain.

In [11]:
# ============================================================
# Section 11: RAG Retrieval
# ============================================================

import numpy as np
import pandas as pd


def retrieve_relevant_documents(query_text, embedding_model, faiss_index, recipe_documents, top_k=5):
    """Embed a query, search FAISS, and return the most relevant recipe documents."""
    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    query_embedding = np.array(query_embedding).astype("float32")

    similarity_scores, document_indices = faiss_index.search(query_embedding, top_k)

    retrieved_rows = []

    for rank, document_index in enumerate(document_indices[0], start=1):
        document = recipe_documents[int(document_index)]
        retrieved_rows.append({
            "rank": rank,
            "document_id": document["document_id"],
            "title": document["title"],
            "similarity_score": round(float(similarity_scores[0][rank - 1]), 4),
            "text": document["text"],
        })

    return pd.DataFrame(retrieved_rows)


def build_rag_context(retrieved_documents_df, max_characters_per_document=1200):
    """Combine retrieved recipe documents into one prompt context block."""
    context_blocks = []

    for _, document_row in retrieved_documents_df.iterrows():
        document_text = document_row["text"][:max_characters_per_document]
        context_blocks.append(
            f"Document {document_row['rank']} - {document_row['title']}\n{document_text}"
        )

    return "\n\n---\n\n".join(context_blocks)


# Confirm retrieval objects exist from earlier sections.
required_retrieval_objects = ["embedding_model", "faiss_index", "recipe_documents"]
missing_objects = [name for name in required_retrieval_objects if name not in globals()]

if missing_objects:
    raise NameError(f"Please run Sections 8-10 first. Missing: {missing_objects}")


sample_user_query = "I have gin, fresh lemon, fresh lime, sugar syrup, mint, and bitters. Recommend a refreshing citrus cocktail."

retrieved_context_df = retrieve_relevant_documents(
    sample_user_query,
    embedding_model,
    faiss_index,
    recipe_documents,
    top_k=5,
)

rag_context = build_rag_context(retrieved_context_df)

print("User query:")
print(sample_user_query)

print("\nRetrieved documents:")
display(retrieved_context_df[["rank", "title", "similarity_score"]])

print("\nRAG context preview:")
print(rag_context[:1500])

User query:
I have gin, fresh lemon, fresh lime, sugar syrup, mint, and bitters. Recommend a refreshing citrus cocktail.

Retrieved documents:


,rank,title,similarity_score
0,1,Alaskan Lemonade,0.7426
1,2,Springtime gin,0.7377
2,3,Louisville Sour cocktail,0.6947
3,4,Short Island Iced Tea Cocktail,0.6930
4,5,Bee's Fizz,0.6903



RAG context preview:
Document 1 - Alaskan Lemonade
Cocktail: Alaskan Lemonade
Glass: Fizz or Highball (8oz to 10oz)
Garnish: Slice of lime

Ingredients:
- 6 cl The Botanist Islay Dry Gin
- 3 cl Yellow (Jaune) Chartreuse liqueur
- 1.5 cl Cointreau triple sec liqueur
- 1 cl Lime juice (freshly squeezed)
- 13.5 cl Carbonated/Sparkling/Selzer mineral water
- 2 Splash 'Simple' sugar syrup (1 sugar to 1 water)
- 2 dash Angostura Aromatic Bitters

Instructions:
Stir and serve over ice. Top with mineral water.

---

Document 2 - Springtime gin
Cocktail: Springtime gin
Glass: Martini (small) glass
Garnish: Sugar rim and amarena cherry

Ingredients:
- 3 cl Monkey 47 Schwarzwald Dry Gin
- 1.5 cl Crème de violette liqueur
- 0.75 cl Lemon juice (freshly squeezed)
- 0.75 cl Orange juice (freshly squeezed)
- 0.5 cl Fabbri Amarena Mixybar cherry syrup
- 0.5 cl 'Simple' sugar syrup (1 sugar to 1 water)
- 4.5 cl Fever-Tree Handpicked Elderflower Tonic

Instructions:
Shake everything but tonic water in 

## Example Output

Example retrieved table:

| rank | title | similarity_score |
|---:|---|---:|
| 1 | Gin Sour | 0.62 |
| 2 | Southside | 0.58 |
| 3 | Gimlet | 0.55 |
| 4 | Tom Collins | 0.53 |
| 5 | Bee's Knees | 0.50 |

The exact results depend on the embedding model and dataset content.

## Short Comments

- We embed the user query with the same model used for recipe documents.
- FAISS returns document indices and similarity scores.
- We convert retrieved rows into text context for the LLM.
- This follows the simple RAG pattern: embed question, search index, build context, then prompt.

# Section 12: LLM Recommendation

This section adds the generation part of RAG.

We will build a prompt manually using:

- the user's available ingredients
- the top ingredient-matching results
- the retrieved recipe context from FAISS
- the user's taste preference

Following the simple RAG notebook style, we load one small instruction model and use one recommendation function to build the prompt and generate the answer.

## Why This Step Is Necessary

Retrieval finds relevant recipe facts, but the final answer should be friendly and personalized.

The LLM turns retrieved context and matching results into a natural-language recommendation.

This section demonstrates prompting without hiding the RAG logic inside a high-level framework.

In [12]:
# ============================================================
# Section 12: LLM Recommendation
# ============================================================

from transformers import pipeline


LLM_MODEL_NAME = "google/flan-t5-small"

# Load a small text-to-text model for teaching.
llm_generator = pipeline("text2text-generation", model=LLM_MODEL_NAME)


def summarize_matching_results_for_prompt(matching_df, max_rows=5):
    """Convert ingredient matching rows into compact prompt text."""
    lines = []

    for _, row in matching_df.head(max_rows).iterrows():
        missing_text = ", ".join(row["missing_ingredients"]) if row["missing_ingredients"] else "none"
        lines.append(f"- {row['title']}: {row['match_percentage']}% match; missing: {missing_text}")

    return "\n".join(lines)


def build_llm_recommendation_prompt(user_preference, available_ingredients, matching_df, rag_context):
    """Build a clear RAG prompt for the mixologist recommendation."""
    available_text = ", ".join(available_ingredients)
    matching_summary = summarize_matching_results_for_prompt(matching_df, max_rows=5)

    return f"""
You are an AI home mixologist.

Use only the retrieved cocktail documents and ingredient matching results below.
Do not claim the user has an ingredient if it appears in the missing ingredient list.
Remember that different juices are different ingredients, and fresh citrus means fresh fruit.

User preference:
{user_preference}

Ingredients the user has:
{available_text}

Ingredient matching results:
{matching_summary}

Retrieved recipe context:
{rag_context}

Recommend 2 cocktails. For each cocktail, explain why it fits, list missing ingredients, and give one short preparation note.
""".strip()


def generate_fallback_recommendation(matching_df):
    """Create a simple recommendation directly from ingredient matching results."""
    lines = ["Recommendation based on ingredient matching:"]

    for _, row in matching_df.head(2).iterrows():
        missing_text = ", ".join(row["missing_ingredients"]) if row["missing_ingredients"] else "no missing ingredients"
        lines.append(f"- {row['title']}: {row['match_percentage']}% match. Missing: {missing_text}.")

    return "\n".join(lines)


def generate_llm_recommendation(prompt, matching_df):
    """Generate a recommendation from the prompt, with a simple fallback."""
    try:
        output = llm_generator(
            prompt,
            max_new_tokens=180,
            do_sample=False,
            truncation=True,
        )
        return output[0]["generated_text"]
    except Exception as error:
        print("LLM generation failed, so the notebook is using the matching fallback.")
        print("Reason:", str(error)[:250])
        return generate_fallback_recommendation(matching_df)


# Confirm needed objects exist from earlier sections.
required_llm_objects = ["user_available_ingredients", "matching_results_df", "rag_context"]
missing_objects = [name for name in required_llm_objects if name not in globals()]

if missing_objects:
    raise NameError(f"Please run Sections 5 and 11 first. Missing: {missing_objects}")


sample_user_preference = "I want something refreshing, citrusy, and not too sweet."

llm_prompt = build_llm_recommendation_prompt(
    sample_user_preference,
    user_available_ingredients,
    matching_results_df,
    rag_context,
)

print("Prompt preview:")
print(llm_prompt[:2000])

print("\nGenerated recommendation:")
llm_recommendation_text = generate_llm_recommendation(llm_prompt, matching_results_df)
print(llm_recommendation_text)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

## Example Output

```text
Generated recommendation:
1. Southside fits because it is refreshing and uses gin, fresh citrus, sugar, and mint. Missing ingredients: none.
2. Gin Sour is another citrus-forward option. Missing ingredients: egg white.
```

If generation fails, the section prints a simple recommendation based on ingredient matching so the notebook can continue.

## Short Comments

- We build the prompt ourselves so the prompting step is visible.
- We include retrieved documents so the LLM answer is grounded in recipe data.
- We include ingredient matching results so the answer considers what the user already owns.
- The code is intentionally closer to a simple teaching RAG notebook: load model, build prompt, generate answer.

# Section 13: Interactive Demo

This section creates a simple Colab interface for the AI Home Mixologist.

The user can enter:

- ingredients they have at home
- a taste preference
- how many recommendations they want

The demo then runs ingredient matching, shopping list generation, best-value analysis, retrieval, and recommendation generation.

## Why This Step Is Necessary

A project becomes easier to understand when users can interact with it.

The demo brings together the full pipeline:

```text
user ingredients -> matching -> shopping list -> best value -> retrieval -> recommendation
```

This also helps us test whether all previous sections work together.

In [ ]:
# ============================================================
# Section 13: Interactive Demo
# ============================================================

import ipywidgets as widgets
from IPython.display import clear_output, display


def parse_user_ingredients_input(ingredients_text):
    """Parse comma-separated user ingredients from a text box."""
    ingredients = [ingredient.strip() for ingredient in ingredients_text.split(",")]
    return [ingredient for ingredient in ingredients if ingredient]


def run_mixologist_pipeline(available_ingredients, user_preference, recommendation_count=5, use_local_llm=False):
    """Run the full AI Home Mixologist pipeline for one user input."""
    top_matches_df = rank_cocktails_by_ingredients(
        cocktails_clean_df,
        available_ingredients,
        top_n=recommendation_count,
    )

    top_shopping_list_df = generate_shopping_list(top_matches_df, top_n=recommendation_count)

    top_best_value_df = analyze_best_value_purchase(
        cocktails_clean_df,
        available_ingredients,
        top_n=10,
    )

    retrieval_query = (
        f"Preference: {user_preference}. "
        f"Available ingredients: {', '.join(available_ingredients)}."
    )

    top_retrieved_df = retrieve_relevant_documents(
        retrieval_query,
        embedding_model,
        faiss_index,
        recipe_documents,
        top_k=5,
    )

    top_rag_context = build_rag_context(top_retrieved_df)
    demo_prompt = build_llm_recommendation_prompt(
        user_preference,
        available_ingredients,
        top_matches_df,
        top_rag_context,
    )

    if use_local_llm:
        recommendation_text = generate_llm_recommendation(demo_prompt, top_matches_df)
    else:
        recommendation_text = generate_fallback_recommendation(top_matches_df)

    return {
        "matches": top_matches_df,
        "shopping_list": top_shopping_list_df,
        "best_value": top_best_value_df,
        "retrieved_documents": top_retrieved_df,
        "recommendation_text": recommendation_text,
    }


# Confirm full pipeline objects exist.
required_demo_objects = [
    "cocktails_clean_df",
    "embedding_model",
    "faiss_index",
    "recipe_documents",
]
missing_objects = [name for name in required_demo_objects if name not in globals()]

if missing_objects:
    raise NameError(f"Please run the previous sections first. Missing: {missing_objects}")


# Build simple Colab widgets.
ingredients_widget = widgets.Textarea(
    value="gin, fresh lime, fresh lemon, sugar syrup, mint leaves, soda water",
    description="Ingredients",
    layout=widgets.Layout(width="100%", height="90px"),
)

preference_widget = widgets.Text(
    value="refreshing and citrusy",
    description="Preference",
    layout=widgets.Layout(width="100%"),
)

recommendation_count_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    step=1,
    description="Top N",
)

use_llm_widget = widgets.Checkbox(
    value=False,
    description="Use local teaching LLM",
)

run_button = widgets.Button(
    description="Recommend Cocktails",
    button_style="success",
)

demo_output = widgets.Output()


def handle_demo_button_click(button):
    """Run the demo when the user clicks the button."""
    with demo_output:
        clear_output()

        available_ingredients = parse_user_ingredients_input(ingredients_widget.value)
        user_preference = preference_widget.value
        recommendation_count = recommendation_count_widget.value
        use_local_llm = use_llm_widget.value

        demo_results = run_mixologist_pipeline(
            available_ingredients,
            user_preference,
            recommendation_count=recommendation_count,
            use_local_llm=use_local_llm,
        )

        print("Top ingredient matches")
        display(demo_results["matches"][[
            "title",
            "match_percentage",
            "matched_count",
            "missing_count",
            "missing_ingredients",
        ]])

        print("\nShopping list")
        display(demo_results["shopping_list"])

        print("\nBest value purchases")
        display(demo_results["best_value"])

        print("\nRetrieved documents")
        display(demo_results["retrieved_documents"][["rank", "title", "similarity_score"]])

        print("\nRecommendation")
        print(demo_results["recommendation_text"])


run_button.on_click(handle_demo_button_click)

display(ingredients_widget, preference_widget, recommendation_count_widget, use_llm_widget, run_button, demo_output)

## Example Output

In Colab, this section displays an interactive form.

After clicking **Recommend Cocktails**, the output area shows:

```text
Top ingredient matches
Shopping list
Best value purchases
Retrieved documents
Recommendation
```

The checkbox controls whether to use the small local teaching LLM. Leaving it unchecked makes the demo faster.

## Short Comments

- The demo uses the same reusable functions built in earlier sections.
- The user can change ingredients without editing code.
- The local LLM option is available, but the fast fallback keeps the demo responsive.
- This section proves that the custom features and RAG pipeline work together.

# Section 14: Evaluation

This final section evaluates the project with simple, beginner-friendly checks.

We evaluate three parts of the system:

- ingredient matching
- retrieval quality
- custom feature sanity checks

This is not a perfect scientific benchmark, but it gives us useful evidence that the notebook works as intended.

## Why This Step Is Necessary

AI projects should be tested, not just demonstrated.

Evaluation helps us catch problems such as:

- matching results that ignore obvious ingredients
- retrieval results unrelated to the query
- shopping lists that fail to merge duplicates
- best-value calculations that return impossible negative counts

A simple evaluation section makes the notebook more complete and more trustworthy.

In [ ]:
# ============================================================
# Section 14: Evaluation
# ============================================================

import pandas as pd


evaluation_cases = [
    {
        "case_name": "Gin citrus home bar",
        "available_ingredients": ["gin", "fresh lemon", "fresh lime", "sugar syrup"],
        "query": "refreshing gin citrus cocktail with lemon or lime",
        "expected_terms": ["gin", "lemon", "lime"],
    },
    {
        "case_name": "Rum tropical home bar",
        "available_ingredients": ["rum", "fresh lime", "pineapple juice", "sugar syrup"],
        "query": "tropical rum cocktail with pineapple and lime",
        "expected_terms": ["rum", "pineapple", "lime"],
    },
    {
        "case_name": "Vodka fruit home bar",
        "available_ingredients": ["vodka", "cranberry juice", "orange juice", "fresh lime"],
        "query": "fruity vodka cocktail with cranberry or orange",
        "expected_terms": ["vodka", "cranberry", "orange"],
    },
]


def text_contains_expected_terms(text, expected_terms):
    """Check whether text contains at least one expected term."""
    normalized_text = normalize_ingredient_name(text)
    return any(normalize_ingredient_name(term) in normalized_text for term in expected_terms)


def evaluate_matching_case(clean_dataframe, available_ingredients, expected_terms, top_k=5):
    """Evaluate whether ingredient matching returns plausible top results."""
    case_matches_df = rank_cocktails_by_ingredients(
        clean_dataframe,
        available_ingredients,
        top_n=top_k,
    )

    combined_top_text = " ".join(
        case_matches_df["title"].astype(str).tolist()
        + case_matches_df["matched_ingredients"].astype(str).tolist()
        + case_matches_df["missing_ingredients"].astype(str).tolist()
    )

    return {
        "matching_hit_at_k": text_contains_expected_terms(combined_top_text, expected_terms),
        "best_match_percentage": float(case_matches_df.iloc[0]["match_percentage"]),
        "average_top_k_match_percentage": round(float(case_matches_df["match_percentage"].mean()), 2),
    }


def evaluate_retrieval_case(query, expected_terms, top_k=5):
    """Evaluate whether retrieved documents contain expected query terms."""
    retrieved_df = retrieve_relevant_documents(
        query,
        embedding_model,
        faiss_index,
        recipe_documents,
        top_k=top_k,
    )

    retrieved_text = " ".join(retrieved_df["text"].astype(str).tolist())

    return {
        "retrieval_hit_at_k": text_contains_expected_terms(retrieved_text, expected_terms),
        "average_similarity_score": round(float(retrieved_df["similarity_score"].mean()), 4),
    }


def run_system_evaluation(evaluation_cases):
    """Run all evaluation cases and return a summary DataFrame."""
    evaluation_rows = []

    for evaluation_case in evaluation_cases:
        matching_metrics = evaluate_matching_case(
            cocktails_clean_df,
            evaluation_case["available_ingredients"],
            evaluation_case["expected_terms"],
            top_k=5,
        )

        retrieval_metrics = evaluate_retrieval_case(
            evaluation_case["query"],
            evaluation_case["expected_terms"],
            top_k=5,
        )

        evaluation_rows.append({
            "case_name": evaluation_case["case_name"],
            **matching_metrics,
            **retrieval_metrics,
        })

    return pd.DataFrame(evaluation_rows)


def run_custom_feature_sanity_checks():
    """Run simple checks for shopping list and best-value analyzer outputs."""
    sanity_messages = []

    sample_matches_df = rank_cocktails_by_ingredients(
        cocktails_clean_df,
        user_available_ingredients,
        top_n=10,
    )
    sample_shopping_df = generate_shopping_list(sample_matches_df, top_n=10)
    sample_best_value_df = analyze_best_value_purchase(
        cocktails_clean_df,
        user_available_ingredients,
        top_n=10,
    )

    shopping_has_no_duplicates = sample_shopping_df["ingredient"].is_unique if not sample_shopping_df.empty else True
    sanity_messages.append({
        "check_name": "Shopping list merges duplicate ingredients",
        "passed": bool(shopping_has_no_duplicates),
    })

    best_value_non_negative = (
        sample_best_value_df["extra_cocktails_unlocked"] >= 0
    ).all() if not sample_best_value_df.empty else True
    sanity_messages.append({
        "check_name": "Best-value unlocked counts are non-negative",
        "passed": bool(best_value_non_negative),
    })

    matching_percent_valid = sample_matches_df["match_percentage"].between(0, 100).all()
    sanity_messages.append({
        "check_name": "Match percentages stay between 0 and 100",
        "passed": bool(matching_percent_valid),
    })

    return pd.DataFrame(sanity_messages)


# Confirm the full pipeline exists before evaluation.
required_evaluation_objects = [
    "cocktails_clean_df",
    "embedding_model",
    "faiss_index",
    "recipe_documents",
    "user_available_ingredients",
]
missing_objects = [name for name in required_evaluation_objects if name not in globals()]

if missing_objects:
    raise NameError(f"Please run the previous sections first. Missing: {missing_objects}")


evaluation_results_df = run_system_evaluation(evaluation_cases)
sanity_checks_df = run_custom_feature_sanity_checks()

print("Evaluation results:")
display(evaluation_results_df)

print("\nCustom feature sanity checks:")
display(sanity_checks_df)

print("\nEvaluation summary:")
print("Matching hit rate:", evaluation_results_df["matching_hit_at_k"].mean())
print("Retrieval hit rate:", evaluation_results_df["retrieval_hit_at_k"].mean())
print("Sanity checks passed:", sanity_checks_df["passed"].mean())

## Example Output

Example evaluation table:

| case_name | matching_hit_at_k | best_match_percentage | retrieval_hit_at_k | average_similarity_score |
|---|---|---:|---|---:|
| Gin citrus home bar | True | 100.00 | True | 0.52 |
| Rum tropical home bar | True | 80.00 | True | 0.49 |
| Vodka fruit home bar | True | 75.00 | True | 0.47 |

Example summary:

```text
Matching hit rate: 1.0
Retrieval hit rate: 1.0
Sanity checks passed: 1.0
```

## Short Comments

- We use small hand-written evaluation cases because this is a teaching notebook.
- Matching evaluation checks whether ingredient-based recommendations contain expected terms.
- Retrieval evaluation checks whether vector search returns documents related to the query.
- Sanity checks verify that custom features produce reasonable outputs.
- A stronger production system would use a larger labeled evaluation set and human review.